# 05.02 - Serving multiple models: memory budgeting & concurrency

**Why.** One `GenAIServer` process can host an LLM, a VLM, and an ASR model at the same time (served
names `llm`, `vlm`, `asr`). That is convenient - one endpoint, three capabilities - but it is also
where memory and concurrency reality bites on a small board. This notebook is about **budgeting** the
three-model server and understanding **what actually happens** when requests arrive at once.

Concept + documented-procedure. Nothing here runs a model; heavy launches are printed strings.
Grounded in tutorial 021 and `core/include/genai/GenAIServer.h`.

## The three models already on the board

Every happy path here uses models that are **already pulled** - no `llima pull` (`llima list`,
2026-07-09):

| Served name | Model ID | Directory |
| --- | --- | --- |
| `llm` | `Qwen3-4B-Instruct-2507-GPTQ-a16w4` | `/media/nvme/llima/models/Qwen3-4B-Instruct-2507-GPTQ-a16w4` |
| `vlm` | `Qwen3-VL-4B-Instruct-GPTQ-a16w4` | `/media/nvme/llima/models/Qwen3-VL-4B-Instruct-GPTQ-a16w4` |
| `asr` | `whisper-small-a16w8` | `/media/nvme/llima/models/whisper-small-a16w8` |

`scripts/serve_multi_model.py` registers exactly these three by default.

## Memory budgeting: every registered model is resident

`add_model(...)` **loads** the model - it does not lazily load on first request. So a three-model
server holds three sets of weights at once. Budget on two axes:

- **Disk** - the board root FS has only **~5.9 GB free of 14 GB** (`df -h /`, 2026-07-09). Weights
  live on `/media/nvme` (larger), but the two `a16w4` 4B models plus the `a16w8` whisper are not
  free. **Always `df -h /` and `df -h /media/nvme` before adding a fresh model.**
- **Runtime memory** - each resident model consumes RAM for weights + KV cache. The quantization
  suffix drives the weight footprint: `a16w4` (4-bit) is ~half of `a16w8` (8-bit). Two 4B `a16w4`
  models is roughly the practical ceiling before you start trading models in and out.

**Budgeting tactic:** if all three do not fit, serve a subset. `serve_multi_model.py` lets you
disable a slot by passing an empty path (`--vlm '' --asr ''` serves LLM only).

In [ ]:
# Cheap, safe precheck (no model run). Printed string; owner runs it on the board.
disk_precheck = "df -h / && df -h /media/nvme && llima list"
print("Before starting a multi-model server:")
print(" ", disk_precheck)
print()
print("Serve a subset if memory is tight, e.g. LLM only:")
print("  serve_multi_model.py --vlm '' --asr ''")

## Model switching vs concurrency - what really happens

Two questions people conflate:

1. **Switching between served models** - a request for `vlm` and the next for `llm` are just routed
   to different resident handles by served name. No reload, because `add_model` already loaded both.
   This is cheap.
2. **Concurrent requests** - here is the honest limit. From tutorial 021's "In Practice": multiple
   models (and even multiple server processes) **share the same MLA hardware gatekeeper**. Running one
   process with several served names does **not multiply hardware throughput** - the MLA is the
   bottleneck and generation work is serialized on it. More served names give you more *capabilities*
   from one endpoint, not more parallel *tokens/second*.

So: switch freely, but do not expect two simultaneous large generations to both run at full speed.
Design clients to tolerate serialization (queueing, timeouts), not to assume parallelism.

## Concurrency guidance for clients

- Set sensible request **timeouts** (the tutorial clients use 120 s) - a queued request waits for the
  MLA.
- Prefer **streaming** (`"stream": true`) so a client sees first tokens (`ttft`) as soon as the MLA
  reaches its request, instead of blocking on the full completion.
- Do not spin up **multiple server processes** to "parallelize" - they load their own copies of the
  weights (more memory) and still contend for the same MLA gatekeeper. One process, multiple served
  names, is the intended shape.
- If you truly need isolation, serve **fewer** models per process and route at the client, mindful of
  the disk/RAM budget above.

### Launch the three-model server (documented, owner-run)

Printed strings; this is the HEAVY step.

In [ ]:
# HEAVY / MANUAL RUN - not executed by tooling. serve_multi_model.py registers
# llm + vlm + asr by default (all three already on the board -> no pull).

# Human, real terminal:
dk_all = "dk /workspace/demo-neat/llima/05-genai-server/scripts/serve_multi_model.py"

# CI / no TTY:
ssh_all = ("timeout 900 ssh -o BatchMode=yes sima@192.168.135.203 "
           "'source $HOME/pyneat/bin/activate; "
           "python /workspace/demo-neat/llima/05-genai-server/scripts/serve_multi_model.py'")

print("[dk]"); print(" ", dk_all); print()
print("[ssh]"); print(" ", ssh_all)

### Exercise all three endpoints (documented, owner-run)

One server, three capabilities. Uses `scripts/client_examples.py` (adapted from the tutorial 021
request clients). These hit a running server; printed strings only.

In [ ]:
# MANUAL RUN - talks to a running server. Printed strings only.
MODALIX_IP = "192.168.135.203"  # replace with your board IP
base = "python /workspace/demo-neat/llima/05-genai-server/scripts/client_examples.py"

calls = {
    "text  -> llm": f"{base} --run text --server-ip {MODALIX_IP} "
                    "'Give me three tips for designing a small REST API.'",
    "image -> vlm": f"{base} --run image --server-ip {MODALIX_IP} scene.jpg 'What is in this image?'",
    "audio -> asr": f"{base} --run transcribe --server-ip {MODALIX_IP} speech.wav",
    "list models":  f"curl http://{MODALIX_IP}:9998/v1/models",
}
for label, cmd in calls.items():
    print(f"[{label}]"); print(" ", cmd); print()

## Expected output

- `GET /v1/models` lists all three served names: `llm`, `vlm`, `asr`.
- `serve_multi_model.py` prints `registered models: llm, vlm, asr` at startup (after loading all
  three - allow time and confirm disk/RAM headroom first).
- Each `client_examples.py` mode streams its result and prints server `ttft` + `tps`:
  - `text` -> REST API tips
  - `image` -> a description of `scene.jpg` (VLM images are converted to RGB server-side)
  - `transcribe` -> the transcript of `speech.wav`
- Under **simultaneous** load, requests complete correctly but are **serialized on the MLA** - total
  throughput does not exceed a single model's, by design.

If startup fails or the process is killed shortly after `add_model`, you are almost certainly out of
memory/disk - drop to a subset (`--vlm '' --asr ''`) and re-check `df -h /media/nvme`.